In [1]:
import pandas as pd

In [2]:
salesdf = pd.read_csv('sales_messy.csv')

In [3]:
customersdf = pd.read_csv('customers.csv')

In [4]:
salesdf.shape

(208, 9)

In [6]:
salesdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB


In [7]:
salesdf.isna().sum()

order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64

In [8]:
salesdf.duplicated().sum()

np.int64(8)

In [9]:
salesdf["country"].unique()

<StringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Poland',          'usa',          'USA',       'Russia']
Length: 12, dtype: str

# Step 2. Clean Data
Remove duplicates, standardize country names, handle missing values, and convert dates.

In [10]:
salesdf = salesdf.drop_duplicates()

In [11]:
salesdf["country"] = (
    salesdf["country"]
    .str.strip()
    .str.title()
)

In [12]:
salesdf["discount"] = salesdf["discount"].fillna(0)

In [13]:
salesdf["unit_price"] = salesdf["unit_price"].fillna(salesdf["unit_price"].median())

In [14]:
salesdf = salesdf.dropna(subset=["customer_id"])

In [16]:
salesdf["order_date"] = pd.to_datetime(salesdf["order_date"])

In [17]:
salesdf[["discount", "unit_price", "customer_id"]].isna().sum()

discount       0
unit_price     0
customer_id    0
dtype: int64

# 3. Enrich Data
Create new columns for revenue and month analysis.
Revenue = Quantity × Unit Price × (1 − Discount)

In [18]:
salesdf["revenue"] = (salesdf["quantity"] *salesdf["unit_price"] *(1 - salesdf["discount"]))

In [19]:
salesdf["month"] = (salesdf["order_date"].dt.to_period("M"))

# 4. Merge Customer Data
Join sales data with customer information using customer_id.

In [21]:
rows_before = len(salesdf)

In [23]:
salesdf = salesdf.merge(customersdf,on="customer_id",how="left")

In [24]:
rows_after = len(salesdf)

In [25]:
rows_before == rows_after

True

# 5. Aggregation Analysis
Calculate total revenue by category, month, and customer segment.

In [32]:
revenue_category = (salesdf.groupby("category")["revenue"].sum().sort_values(ascending=False).reset_index())
revenue_category

,category,revenue
0,Laptops,161187.4000
1,Phones,63403.4000
2,Monitors,58295.5500
3,Accessories,10323.5465


In [33]:
revenue_month = (salesdf.groupby("month")["revenue"].sum().sort_values(ascending=False).reset_index())
revenue_month

,month,revenue
0,2025-07,42529.4260
1,2025-10,33697.7500
2,2025-08,30827.4315
3,2025-04,26456.2405
4,2025-06,24754.8375
5,2025-05,23633.5065
6,2025-12,22739.7510
7,2025-03,19836.5880
8,2025-02,19631.0780
9,2025-11,19117.3905


In [34]:
revenue_segment = (salesdf.groupby("segment")["revenue"].sum().sort_values(ascending=False).reset_index())
revenue_segment

,segment,revenue
0,Consumer,174850.8365
1,Education,77782.0320
2,Business,40577.0280


# Conclusions

1. Laptops generated the highest revenue of 161,187.40, accounting for 54.97% of total revenue. This means more than half of all sales revenue came from the laptop category.

2. The best-performing month was 2025-07, with total revenue of 42,529.43.

3. The Consumer segment was the leading customer segment, generating 174,850.84 in revenue, significantly higher than Education (77,782.03) and Business (40,577.03).

4. Accessories produced the lowest revenue at 10,323.55, while Laptops produced 161,187.40. The difference between these categories was 150,863.85.

5. A surprising observation is that the Consumer segment alone generated more revenue than the Education and Business segments combined, highlighting its importance to overall sales performance.